In [ ]:
!pip install tensorflow transformers opencv-python-headless

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical

# Load FER2013 dataset from Google Drive
data = pd.read_csv('/content/drive/MyDrive/fer2013.csv')

# Preprocess the data
def preprocess_data(data):
    images = np.array([np.fromstring(pixels, dtype=int, sep=' ').reshape(48, 48, 1) for pixels in data['pixels']])
    images = np.repeat(images, 3, axis=-1)  # Convert grayscale to RGB for EfficientNet
    images = images / 255.0  # Normalize
    labels = to_categorical(data['emotion'], num_classes=7)  # One-hot encode emotions
    return images, labels

train_data = data[data['Usage'] == 'Training']
val_data = data[data['Usage'] == 'PublicTest']

X_train, y_train = preprocess_data(train_data)
X_val, y_val = preprocess_data(val_data)

# Data augmentation
datagen = ImageDataGenerator(horizontal_flip=True, rotation_range=10)
train_gen = datagen.flow(X_train, y_train, batch_size=32)
val_gen = ImageDataGenerator().flow(X_val, y_val, batch_size=32)


In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Multiply, Input
from tensorflow.keras.models import Model

# Attention mechanism
def attention_block(x):
    attention = Dense(1280, activation='relu')(x)  # Match EfficientNet output shape
    attention = Dense(1280, activation='sigmoid')(attention)  # Attention weights
    return Multiply()([x, attention])  # Element-wise multiplication with attention weights

# Build EfficientNet model with attention
def build_emotion_model():
    base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(48, 48, 3))
    x = base_model.output
    x = GlobalAveragePooling2D()(x)

    # Apply attention block
    attention_output = attention_block(x)

    # Final classification layer
    output = Dense(7, activation='softmax')(attention_output)

    model = Model(inputs=base_model.input, outputs=output)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

emotion_model = build_emotion_model()
emotion_model.summary()


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)  │ (None, 48, 48, 3)      │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ rescaling (Rescaling)     │ (None, 48, 48, 3)      │              0 │ input_layer[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ normalization             │ (None, 48, 48, 3)      │              7 │ rescaling[0][0]        │
│ (Normalization)           │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ rescaling_1 (Rescaling)   │ (None, 48, 48, 3)      │              0 │ normalization[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ stem_conv_pad             │ (None, 49, 49, 3)      │              0 │ rescaling_1[0][0]      │
│ (ZeroPadding2D)           │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ stem_conv (Conv2D)        │ (None, 24, 24, 32)     │            864 │ stem_conv_pad[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ stem_bn                   │ (None, 24, 24, 32)     │            128 │ stem_conv[0][0]        │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ stem_activation           │ (None, 24, 24, 32)     │              0 │ stem_bn[0][0]          │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1a_dwconv            │ (None, 24, 24, 32)     │            288 │ stem_activation[0][0]  │
│ (DepthwiseConv2D)         │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1a_bn                │ (None, 24, 24, 32)     │            128 │ block1a_dwconv[0][0]   │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1a_activation        │ (None, 24, 24, 32)     │              0 │ block1a_bn[0][0]       │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1a_se_squeeze        │ (None, 32)             │              0 │ block1a_activation[0]… │
│ (GlobalAveragePooling2D)  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1a_se_reshape        │ (None, 1, 1, 32)       │              0 │ block1a_se_squeeze[0]… │
│ (Reshape)                 │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1a_se_reduce         │ (None, 1, 1, 8)        │            264 │ block1a_se_reshape[0]… │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1a_se_expand    

 Total params: 7,337,898 (27.99 MB)

 Trainable params: 7,295,875 (27.83 MB)

 Non-trainable params: 42,023 (164.16 KB)

In [ ]:
# Train the model
history = emotion_model.fit(train_gen, epochs=25, validation_data=val_gen)

# Save the trained model
emotion_model.save('/content/drive/MyDrive/emotion_recognition_model.h5')


Epoch 1/25


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


898/898 ━━━━━━━━━━━━━━━━━━━━ 663s 683ms/step - accuracy: 0.4113 - loss: 1.5331 - val_accuracy: 0.5235 - val_loss: 1.3009
Epoch 2/25
898/898 ━━━━━━━━━━━━━━━━━━━━ 618s 679ms/step - accuracy: 0.5558 - loss: 1.1783 - val_accuracy: 0.5414 - val_loss: 1.2530
Epoch 3/25
898/898 ━━━━━━━━━━━━━━━━━━━━ 605s 674ms/step - accuracy: 0.5843 - loss: 1.0975 - val_accuracy: 0.5874 - val_loss: 1.1139
Epoch 4/25
898/898 ━━━━━━━━━━━━━━━━━━━━ 605s 673ms/step - accuracy: 0.6145 - loss: 1.0319 - val_accuracy: 0.5770 - val_loss: 1.1346
Epoch 5/25
898/898 ━━━━━━━━━━━━━━━━━━━━ 631s 684ms/step - accuracy: 0.6284 - loss: 0.9927 - val_accuracy: 0.6119 - val_loss: 1.0637
Epoch 6/25
898/898 ━━━━━━━━━━━━━━━━━━━━ 619s 681ms/step - accuracy: 0.6543 - loss: 0.9345 - val_accuracy: 0.5837 - val_loss: 1.1254
Epoch 7/25
898/898 ━━━━━━━━━━━━━━━━━━━━ 607s 675ms/step - accuracy: 0.6581 - loss: 0.9105 - val_accuracy: 0.5996 - val_loss: 1.0936
Epoch 8/25
898/898 ━━━━━━━━━━━━━━━━━━━━ 623s 676ms/step - accuracy: 0.6759 - loss: 0.87

In [ ]:
# Train the model
history = emotion_model.fit(train_gen, epochs=50, validation_data=val_gen)

# Save the trained model
emotion_model.save('/content/drive/MyDrive/emotion_recognition_model.h5')

Epoch 1/50


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


898/898 ━━━━━━━━━━━━━━━━━━━━ 614s 633ms/step - accuracy: 0.4061 - loss: 1.5355 - val_accuracy: 0.5216 - val_loss: 1.2597
Epoch 2/50
898/898 ━━━━━━━━━━━━━━━━━━━━ 612s 622ms/step - accuracy: 0.5604 - loss: 1.1762 - val_accuracy: 0.5626 - val_loss: 1.1539
Epoch 3/50
898/898 ━━━━━━━━━━━━━━━━━━━━ 558s 617ms/step - accuracy: 0.5883 - loss: 1.0935 - val_accuracy: 0.5857 - val_loss: 1.1351
Epoch 4/50
898/898 ━━━━━━━━━━━━━━━━━━━━ 555s 618ms/step - accuracy: 0.6055 - loss: 1.0461 - val_accuracy: 0.5584 - val_loss: 1.2143
Epoch 5/50
898/898 ━━━━━━━━━━━━━━━━━━━━ 554s 617ms/step - accuracy: 0.6329 - loss: 0.9804 - val_accuracy: 0.5823 - val_loss: 1.1226
Epoch 6/50
898/898 ━━━━━━━━━━━━━━━━━━━━ 559s 614ms/step - accuracy: 0.6490 - loss: 0.9489 - val_accuracy: 0.6074 - val_loss: 1.0785
Epoch 7/50
898/898 ━━━━━━━━━━━━━━━━━━━━ 573s 626ms/step - accuracy: 0.6560 - loss: 0.9129 - val_accuracy: 0.6082 - val_loss: 1.0694
Epoch 8/50
898/898 ━━━━━━━━━━━━━━━━━━━━ 548s 610ms/step - accuracy: 0.6817 - loss: 0.86

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive
